# TAME-Fusion Training Dynamics & MoE Gating Analysis

Evaluates training dynamics from `scripts/tame_fusion_training_dynamics.py`.
Loads artifacts from `results/` and plots:
1. **Train vs. Validation Loss** (BCE) and **Validation ROC-AUC** over epochs
2. **MoE Gate Contributions** over epochs (SEG branch vs. Descriptor branch)
3. **Gate Distributions**: How are gates distributed per molecule/position?
4. **Gate Entropy Diagnostic**: Are gates soft (uniform) or hard (0/1)?

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

here = Path.cwd().resolve()
workspace_root = next((p for p in [here, *here.parents] if (p / "models").exists()), here)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

In [ ]:
# ====================================================================
# 1. Load artifacts
# ====================================================================
results_dir = workspace_root / "benchmarking" / "results"
# results_dir = Path("/path/to/hpc/output")  # override when loading from cluster

with open(results_dir / "tame_fusion_dynamics_histories.json") as f:
    all_histories = json.load(f)

gates_data = np.load(results_dir / "tame_fusion_dynamics_gates.npz")
seg_gates  = gates_data["seg_gates"]   # (N_val, hidden_dim)
desc_gates = gates_data["desc_gates"]  # (N_val, hidden_dim)

print(f"Loaded {len(all_histories)} seeds.")
print(f"Gate arrays: seg={seg_gates.shape}, desc={desc_gates.shape}")

In [ ]:
# ====================================================================
# 2. Helper: pad unequal epoch lengths for aggregation
# ====================================================================
def pad_histories(hist_list, key):
    max_len = max(len(h[key]) for h in hist_list)
    padded = []
    for h in hist_list:
        arr = np.array(h[key], dtype=np.float32)
        pad_width = max_len - len(arr)
        padded.append(np.pad(arr, (0, pad_width), constant_values=np.nan))
    matrix = np.stack(padded)
    return np.nanmean(matrix, axis=0), np.nanstd(matrix, axis=0)

In [ ]:
# ====================================================================
# 3. Plot: Loss Dynamics & ROC-AUC
# ====================================================================
plt.style.use("default")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "axes.grid": True, "grid.alpha": 0.3})

train_mean, train_std = pad_histories(all_histories, "train_loss")
val_mean,   val_std   = pad_histories(all_histories, "val_loss")
auc_mean,   auc_std   = pad_histories(all_histories, "val_auc")
epochs = np.arange(1, len(train_mean) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_mean, label="Train Loss (BCE)", color="#2563eb", linewidth=2)
axes[0].fill_between(epochs, train_mean - train_std, train_mean + train_std,
                      color="#2563eb", alpha=0.2)
axes[0].plot(epochs, val_mean, label="Validation Loss (BCE)", color="#dc2626", linewidth=2)
axes[0].fill_between(epochs, val_mean - val_std, val_mean + val_std,
                      color="#dc2626", alpha=0.2)

# Mark early stopping epoch for each seed
for h in all_histories:
    axes[0].axvline(len(h["val_loss"]), color="gray", linestyle=":", alpha=0.35, linewidth=1)

axes[0].set_title("Loss Dynamics (BCE)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(epochs, auc_mean, label="Validation ROC-AUC", color="#059669", linewidth=2)
axes[1].fill_between(epochs, auc_mean - auc_std, auc_mean + auc_std,
                      color="#059669", alpha=0.2)
for h in all_histories:
    axes[1].axvline(len(h["val_loss"]), color="gray", linestyle=":", alpha=0.35, linewidth=1)

axes[1].set_title("Validation Metric (ROC-AUC)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("ROC-AUC", fontsize=11)
axes[1].legend(loc="lower right")
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("TAME-Fusion Training Dynamics (Mean ± Std over 5 Seeds)",
             y=1.05, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# 4. Plot: MoE Gate Contributions over epochs
# ====================================================================
seg_mean,  seg_std  = pad_histories(all_histories, "seg_gate")
desc_mean, desc_std = pad_histories(all_histories, "desc_gate")

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(epochs, seg_mean,  label="SEG Branch (Graph+Text)", color="#3b82f6", linewidth=2)
ax.fill_between(epochs, seg_mean - seg_std, seg_mean + seg_std, color="#3b82f6", alpha=0.2)

ax.plot(epochs, desc_mean, label="Descriptor Branch",       color="#f59e0b", linewidth=2)
ax.fill_between(epochs, desc_mean - desc_std, desc_mean + desc_std, color="#f59e0b", alpha=0.2)

ax.axhline(0.5, color="black", linestyle="--", alpha=0.5, label="Uniform (0.50)")

ax.set_title("TAME-Fusion MoE Gate Contributions per Epoch", fontsize=14, fontweight="bold", pad=10)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Mean Gate Weight", fontsize=12)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Gate Distribution (Modality Collapse Check)

Shows whether the model assigns individual hidden-dimension positions soft intermediate weights
or collapses to hard 0/1 routing (each position fully owned by one modality).

- **Spike at 0 and 1** → hard routing (each position entirely owned by one branch)
- **Uniform spread** → soft routing (blended contributions)

Hard routing with ~50/50 balance across positions is **not modality collapse** — both branches
contribute, just at different hidden dimensions. However it can impede gradient flow to the
unused expert at each position. Use `gate_entropy_weight > 0` to encourage softer routing.

In [ ]:
# ====================================================================
# 5. Plot: Gate value distributions (validation set, last seed)
# ====================================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sns.histplot(np.array(seg_gates).flatten(),  bins=30, ax=axes[0], color="#3b82f6", kde=True, legend=False)
axes[0].set_title("SEG Branch Gate Distribution")
axes[0].set_xlabel("Gate Weight")

sns.histplot(np.array(desc_gates).flatten(), bins=30, ax=axes[1], color="#f59e0b", kde=True, legend=False)
axes[1].set_title("Descriptor Branch Gate Distribution")
axes[1].set_xlabel("Gate Weight")

plt.suptitle("Are the experts collapsed? (Validation Set, Last Seed)",
             y=1.05, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Gate Entropy Diagnostic

Per-position gate entropy: $H_i = -\sum_k g_{ik} \log g_{ik}$

- **Max entropy** (uniform): $\log(2) \approx 0.693$ — fully soft routing
- **Zero entropy** (hard): 0 — one expert takes all weight

A histogram concentrated near 0 confirms hard routing; near 0.693 confirms soft routing.
The balance loss only regulates the mean gate, so hard routing with balanced mean is still possible.

In [ ]:
# ====================================================================
# 6. Gate entropy diagnostic
# ====================================================================
eps = 1e-8
g_s = np.array(seg_gates)   # (N_val, hidden_dim)
g_d = np.array(desc_gates)
ent = -(g_s * np.log(g_s + eps) + g_d * np.log(g_d + eps))  # (N_val, hidden_dim)

max_entropy = np.log(2)  # 2 experts

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(ent.flatten(), bins=60, kde=True, ax=ax, color="#6366f1", legend=False)
ax.axvline(max_entropy, color="#dc2626", linestyle="--", linewidth=1.5,
           label=f"Max entropy (log 2 ≈ {max_entropy:.3f})")
ax.axvline(0.0, color="#f59e0b", linestyle="--", linewidth=1.5,
           label="Hard routing (entropy = 0)")
ax.set_xlabel("Per-position gate entropy", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Gate Entropy Distribution (Validation Set, Last Seed)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

mean_ent = ent.flatten().mean()
print(f"Mean gate entropy: {mean_ent:.4f}  (max = {max_entropy:.4f}, {100*mean_ent/max_entropy:.1f}% of max)")
plt.tight_layout()
plt.show()